## Plotting STAPPS

Imports

In [1]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import numpy as np
from preprocessing import Preprocessor
from loaders import ParticipantDataLoader
from plot_defs import plot_steps_over_time, plot_cycle_markers, plot_all_event_periods, plot_weekends_spans, load_plot_data

from statistics_def import get_hr_zones

mpl.use("QtAgg")

In [2]:
part_ids = [14]

In [3]:
pat_ids, data = load_plot_data(part_ids, ("steps", "hr", "cycles"))  
steps_data = data.get("steps")
heart_rate_data = data.get("hr")
cycle_data = data.get("cycles")

preprocessor = Preprocessor()

daily_stats = preprocessor.get_daily_hr_stats(part_ids, heart_rate_data)



In [5]:
from plot_defs import plot_info


fig, ax = plt.subplots(figsize=(15, 5))

n_parts = len(part_ids)
nrows = 2            # two rows: top = HRV plot, bottom = user-defined plot
ncols = n_parts      # one column per participant

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(10 * ncols, 5 * nrows),
    squeeze=False, 
    sharex="col"
)

for i, part_id in enumerate(part_ids):
    # Plot heart rate data
    plot_steps_over_time(axes[0, i], 
                         steps_data[steps_data["participant_id"] == part_id],
                         "Step Data")

    plot_info(axes[0, i],
              cycle_data[cycle_data["participant_id"] == part_id], steps_data[steps_data["participant_id"] == part_id])
    twin_ax = axes[0, i].twinx()
    # twin_ax.plot(heart_rate_data[heart_rate_data["participant_id"] == part_id]["dateTime"],
    #              heart_rate_data[heart_rate_data["participant_id"] == part_id]["bpm"],
    #              color='orange', zorder= 0, alpha = 0.4, label='Heart Rate')

    ds = daily_stats[daily_stats["participant_id"] == part_id]
    twin_ax.plot(ds["dateTime"], ds["daily_min"], color='green', alpha=0.6, label='Daily Min')
    twin_ax.plot(ds["dateTime"], ds["daily_median"], color='red', alpha=0.8, label='Daily Median')
    twin_ax.plot(ds["dateTime"], ds["daily_max"], color='purple', alpha=0.6, label='Daily Max')
    twin_ax.legend(loc='upper right')
%matplotlib qt
plt.show()

In [6]:
for i, part_id in enumerate(part_ids):
    part_heart_data = heart_rate_data[heart_rate_data["participant_id"] == part_id]
    hr_zone, zone_thresholds = get_hr_zones(part_heart_data, cycle_data)

    def plot_hr_zones(ax, part_heart_data, zone_thresholds, cmap=None):
        if cmap is None:
            cmap = mpl.colormaps.get_cmap("RdYlGn_r") if hasattr(mpl, "colormaps") else mpl.cm.get_cmap("RdYlGn_r")

        finite_bpm = part_heart_data["bpm"].dropna()
        max_bpm = finite_bpm.max() if not finite_bpm.empty else 200.0

        lower = 0
        num_zones = len(zone_thresholds)
        zone_tick_positions = []
        zone_tick_labels = []

        ax_zone = ax.twinx()

        for idx, (key, upper) in enumerate(zone_thresholds.items()):
            upper_val = max_bpm if not np.isfinite(upper) else float(upper)
            frac = idx / max(1, num_zones - 1)

            ax.axhspan(lower, upper_val, xmin=0, xmax=1, alpha=0.5, color=cmap(frac))

            if key != "zero":
                ax.axhline(y=upper_val, color="black", linestyle="--", linewidth=0.5)
                zone_tick_positions.append(upper_val)
                zone_tick_labels.append(f"{key} ({upper_val:.0f})")

            lower = upper_val

        ax_zone.set_ylim(ax.get_ylim())
        ax_zone.set_yticks(zone_tick_positions)
        ax_zone.set_yticklabels(zone_tick_labels, fontsize=8)

        return ax_zone
%matplotlib qt
plt.show()


print(hr_zone.head())

{'min': np.float64(42.0), 'median': np.float64(73.0), 'max': np.float64(163.0)}
   participant_id            dateTime    bpm   zone  confidence
0              14 2025-11-25 14:44:34   91.0  zone1         1.0
1              14 2025-11-25 14:44:36   87.0  zone1         1.0
2              14 2025-11-25 14:44:38  115.0  zone3         1.0
3              14 2025-11-25 14:44:40  109.0  zone2         1.0
4              14 2025-11-25 14:44:42  100.0  zone2         1.0


In [ ]:
for i, part_id in enumerate(part_ids):
    part_heart_data = heart_rate_data[heart_rate_data["participant_id"] == part_id]
    hr_data, zone_thresholds = get_hr_zones(part_heart_data, cycle_data)
    data_merge = preprocessor.merge_cadence_with_hr(steps_data[["dateTime", "cadence"]], hr_data)
# Median cadence per HR zone over time
    data_merge["date"] = data_merge["dateTime"].dt.date
    median_cadence_by_zone = (
        data_merge.groupby(["date", "zone"])["cadence"]
        .median()
        .reset_index()
    )
    median_cadence_by_zone = median_cadence_by_zone.sort_values(["zone", "date"])
    median_cadence_by_zone["cadence"] = (
        median_cadence_by_zone.groupby("zone")["cadence"]
        .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    )
    fig, ax = plt.subplots(figsize=(15, 5))

    for zone, group in median_cadence_by_zone.groupby("zone"):
        ax.plot(group["date"], group["cadence"], label=zone)
        print(median_cadence_by_zone)

    legend = ax.legend(title="HR Zone", loc="upper right")
    plt.legend(fontsize=8)



{'min': np.float64(42.0), 'median': np.float64(73.0), 'max': np.float64(163.0)}
           date     zone    cadence
0    2025-11-25  zone1.0   0.000000
3    2025-11-26  zone1.0   0.000000
7    2025-11-27  zone1.0   0.000000
11   2025-11-28  zone1.0   0.000000
14   2025-11-29  zone1.0   0.000000
..          ...      ...        ...
268  2026-02-08  zone5.0  73.666667
273  2026-02-09  zone5.0  86.666667
296  2026-02-16  zone5.0  82.666667
305  2026-02-22  zone5.0  48.333333
310  2026-02-23  zone5.0  32.666667

[314 rows x 3 columns]
           date     zone    cadence
0    2025-11-25  zone1.0   0.000000
3    2025-11-26  zone1.0   0.000000
7    2025-11-27  zone1.0   0.000000
11   2025-11-28  zone1.0   0.000000
14   2025-11-29  zone1.0   0.000000
..          ...      ...        ...
268  2026-02-08  zone5.0  73.666667
273  2026-02-09  zone5.0  86.666667
296  2026-02-16  zone5.0  82.666667
305  2026-02-22  zone5.0  48.333333
310  2026-02-23  zone5.0  32.666667

[314 rows x 3 columns]
        

: 